# Differential Gene Expression Analysis

##### Franziska Niemeyer, 2026-06-04

In [ ]:
import decoupler
import pandas as pd
import sys
import os
import pickle as pkl
import pydeseq2
import scanpy as sc
import anndata as ad
import numpy as np
import plotnine as p9
from plotnine.layer import layer
from plotnine.options import figure_size
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

sc.set_figure_params(color_map='viridis_r', dpi_save=600, vector_friendly=True, fontsize=12)
color_palette = "Set1"

In [ ]:
WORKING_DIR = "."
ADATA = "../../quality_control/external-cohort/pre-processing/adata.h5ad"
OUT_DIR = os.path.join(WORKING_DIR, "figures")
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)

sc.settings.figdir = OUT_DIR

#### Load and filter the data

In [ ]:
adata = ad.read_h5ad(ADATA)

#### Preprocess the data
Grouping the data by patient, histology, and PFI.
Summarize medium and long PFI as long.

In [ ]:
adata.obs.outcome.value_counts()

In [ ]:
adata = adata[~adata.obs['outcome'].isin(['control tissue'])].copy()

In [ ]:
adata.obs['annotation'] = adata.obs['outcome'].astype(str) + " - " + adata.obs['patient'].astype(str) + " - " + adata.obs['histology'].astype(str)

In [ ]:
adata.obs.histology.value_counts()

In [ ]:
adata.obs['sample'] = adata.obs['sample'].dropna().astype(str)
adata.obs['PFI'] = adata.obs['outcome'].replace({'HC-short': 'short' , 'LC-short': 'short', 'HC-long': 'long', 'LC-long': 'long'}).dropna().astype(str)
adata.obs['patient'] = adata.obs['patient'].dropna().astype(str)
adata.obs['histology'] = adata.obs['histology'].replace({'Ovarian stroma': 'Stroma', 'Tumor epithelium and ovarian stroma': 'Tumor epithelium and adnexal stroma'}).dropna().astype(str)
adata = adata[~adata.obs['PFI'].isna()].copy()
adata.obs

In [ ]:
adata_backup = adata.copy()

In [ ]:
print(adata.obs["patient"].value_counts())

In [ ]:
sc.pp.log1p(adata)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='PFI', method='wilcoxon')

In [ ]:
sc.pl.rank_genes_groups(adata, n_genes=20, key='rank_genes_groups')

In [ ]:
sc.pl.rank_genes_groups_matrixplot(adata, n_genes=10, groupby='PFI')

In [ ]:
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(
    adata,
    color=["PFI", "C3", "IFI27", "BST2"],
    add_outline=True,
    legend_loc="on data",
    legend_fontsize=12,
    legend_fontoutline=2,
    frameon=False,
    title=["PFI", "C3 expression", "IFI27 expression", "BST2 expression"],
)